# Day 11 · Phác thảo sampling và gold set cho bốn camera

**Công cụ hỗ trợ tùy chọn.** Bài gán nhãn và bài nộp chính vẫn ở CVAT và `submission/` trong repo Student. Notebook này giúp bạn tự kiểm một đề xuất cho **tình huống giả lập**, không tạo nhãn, không chấm điểm và không có đáp án mẫu.

Bài ảnh ADASIND trong repo chỉ có **một camera**. Bảng bốn camera dưới đây hoàn toàn giả định để luyện lập kế hoạch SVM/360; không suy rằng nó đo được lỗi cross-camera, free-space hay parking line của ADASIND. Đọc `docs/10-svm360-reading-vi.md` và luật gán nhãn trước khi giải thích kết quả. Chạy các ô từ trên xuống bằng Shift+Enter hoặc nút ▶. Không cần cài thêm thư viện.


## 1 · Đọc tình huống trước khi chọn số

Giả sử một hệ thống có camera **trước, sau, trái, phải**, tổng **50.000 frame** và chỉ đủ ngân sách để đưa **200 frame** vào vòng review đầu tiên. Mỗi frame trong bảng giả lập được gán đúng **một** lát cắt `normal` hoặc `hard` để cộng ngân sách được; thực tế các ca khó có thể đồng thời thuộc nhiều nhóm, nên phải lưu nhiều tag và kiểm trùng khi lập tập thật.

- `normal`: cảnh thông thường trong tình huống giả lập.
- `hard`: có ít nhất một ca cần soi kỹ, ví dụ seam giữa camera, ánh sáng yếu hoặc vạch dễ nhầm giữa **vạch ô đỗ** và **vạch giao thông khác**. Nêu rõ loại ca bạn ưu tiên trong cột `risk`/`rationale` của CSV.

Số lượng ở ô kế tiếp chỉ là giả định của bài tập. Chúng không phải tần suất hay nhãn được xác minh trong dữ liệu lớp.


In [ ]:
CAMERAS = ("front", "rear", "left", "right")
SLICES = ("normal", "hard")
BUDGET = 200
SCENARIO = {
    "front": {"normal": 9700, "hard": 2800},
    "rear": {"normal": 9800, "hard": 2700},
    "left": {"normal": 9950, "hard": 2550},
    "right": {"normal": 10000, "hard": 2500},
}
assert sum(sum(row.values()) for row in SCENARIO.values()) == 50_000
assert all(sum(row.values()) == 12_500 for row in SCENARIO.values())
print("Tình huống giả lập: 50.000 frame; ngân sách review:", BUDGET)
print("camera | " + " | ".join(SLICES) + " | tổng")
for camera in CAMERAS:
    row = SCENARIO[camera]
    print(camera + " | " + " | ".join(str(row[s]) for s in SLICES) + " | " + str(sum(row.values())))


## 2 · Tự phân bổ ngân sách

Thay từng `None` bằng một **số nguyên dương**. Bạn có thể dồn nhiều frame cho lát cắt cần kiểm kỹ, nhưng phải giải thích lựa chọn. Cần có mẫu `normal` và `hard` ở **mọi camera**; đây là ràng buộc học tập của bài này, không phải chuẩn lấy mẫu ngoài thực tế. Tổng phải đúng 200. Đừng dùng số giả lập làm kết luận về rủi ro hay hiệu năng của một camera thật.


In [ ]:
# Điền số frame bạn chọn. Giữ nguyên tên camera/lát cắt.
ALLOCATION = {
    "front": {"normal": None, "hard": None},
    "rear": {"normal": None, "hard": None},
    "left": {"normal": None, "hard": None},
    "right": {"normal": None, "hard": None},
}


In [ ]:
def check_allocation(allocation):
    assert set(allocation) == set(CAMERAS), "Thiếu hoặc thừa camera."
    for camera in CAMERAS:
        assert set(allocation[camera]) == set(SLICES), f"Thiếu hoặc thừa lát cắt: {camera}."
        for slice_name in SLICES:
            chosen = allocation[camera][slice_name]
            assert type(chosen) is int and chosen > 0, f"{camera}/{slice_name}: cần số nguyên dương."
            assert chosen <= SCENARIO[camera][slice_name], f"{camera}/{slice_name}: vượt số frame sẵn có."
    total = sum(sum(row.values()) for row in allocation.values())
    assert total == BUDGET, f"Tổng đang là {total}; cần đúng {BUDGET}."
    return total

if any(value is None for row in ALLOCATION.values() for value in row.values()):
    print("CHƯA ĐIỀN: thay mọi None ở ô ALLOCATION rồi chạy lại ô này.")
else:
    print("ĐỦ ĐIỀU KIỆN CẤU TRÚC: tổng", check_allocation(ALLOCATION), "frame.")
    print("camera | " + " | ".join(SLICES) + " | tổng")
    for camera in CAMERAS:
        row = ALLOCATION[camera]
        print(camera + " | " + " | ".join(str(row[s]) for s in SLICES) + " | " + str(sum(row.values())))


**Sau khi ô kiểm tra qua:** Chuyển tám số sang `submission/45_sampling_plan.csv`, tự điền `risk`/`rationale` cho từng dòng. Trong `45_review_plan.md`, giải thích nguồn bằng chứng cần trước khi chọn frame thật và cách tránh lấy hàng loạt frame sát nhau rồi gọi đó là nhiều ca độc lập. 200 frame theo kế hoạch giúp *tìm loại lỗi cần soi*; nó chưa chứng minh tỷ lệ lỗi, độ phủ ngoài thực tế hay độ đúng của nhãn.


## 3 · Đề xuất gold set theo camera

Với **mỗi camera**, hãy nêu ít nhất một loại ca nên kiểm chứng riêng (ví dụ seam, ánh sáng yếu, vạch mơ hồ), dấu hiệu khiến ca đó khó, và cách đưa vào gold set *sau này*. Một quy trình hợp lý cần ít nhất: người gán nhãn độc lập, người phân xử theo guideline phiên bản ghi rõ, lưu lý do quyết định và xem lại khi guideline/camera thay đổi. Hãy ghi ai chịu trách nhiệm và khi nào cần làm mới; **chưa** có frame ID hay nhãn đã được phân xử trong tình huống giả lập này.

Bản `teaching reference` của lab dùng để học và so sánh, **không tự trở thành gold set**. Thống kê từ một file CVAT của chính bạn cũng không chứng minh correctness. Viết phần đề xuất và giới hạn này trong `submission/46_gold_set_plan.md`; notebook không phải file nộp.


## 4 · Tùy chọn: xem cấu trúc export CVAT của chính bạn

Chỉ dùng khi bạn có quyền đưa file vào môi trường đang chạy. Ô sau **mặc định bỏ qua**. Nếu chạy trên Colab, bạn phải tự upload XML/ZIP bằng giao diện Colab rồi điền đường dẫn vào `CVAT_PATH`; notebook không tự upload, tải repo, gọi API hoặc dùng token. Nếu file không được phép lên dịch vụ ngoài, chạy notebook cục bộ hoặc bỏ qua bước này. Kết quả chỉ đếm image/shapes/label trong export **CVAT for images**; hãy kiểm lại hình học và ngữ nghĩa trong CVAT.


In [ ]:
from collections import Counter
from pathlib import Path
from xml.etree import ElementTree
from zipfile import ZipFile

CVAT_PATH = ""  # Tùy chọn: đường dẫn XML hoặc ZIP của chính bạn. Để trống để bỏ qua.
MAX_XML_BYTES = 20 * 1024 * 1024
SHAPE_TAGS = {"box", "polygon", "polyline", "points", "ellipse", "mask"}

def read_cvat_xml(path_text):
    path = Path(path_text).expanduser()
    if path.suffix.lower() == ".zip":
        with ZipFile(path) as archive:
            candidates = [item for item in archive.infolist()
                          if not item.is_dir() and Path(item.filename).name == "annotations.xml"]
            if len(candidates) != 1:
                raise ValueError("ZIP cần đúng một file annotations.xml của CVAT.")
            item = candidates[0]
            if item.file_size > MAX_XML_BYTES:
                raise ValueError("annotations.xml quá lớn để xem trong notebook.")
            with archive.open(item) as stream:
                xml_bytes = stream.read(MAX_XML_BYTES + 1)
    elif path.suffix.lower() == ".xml":
        if path.stat().st_size > MAX_XML_BYTES:
            raise ValueError("XML quá lớn để xem trong notebook.")
        xml_bytes = path.read_bytes()
    else:
        raise ValueError("Chọn file .xml hoặc .zip từ export CVAT for images.")
    if len(xml_bytes) > MAX_XML_BYTES:
        raise ValueError("XML vượt giới hạn đọc.")
    root = ElementTree.fromstring(xml_bytes)
    if root.tag != "annotations":
        raise ValueError("Đây không phải XML annotations của CVAT for images.")
    return root

def summarize_cvat(root):
    images = root.findall("image")
    label_counts = Counter()
    shape_counts = Counter()
    for image in images:
        for shape in image:
            if shape.tag in SHAPE_TAGS:
                shape_counts[shape.tag] += 1
                label_counts[shape.attrib.get("label", "(thiếu label)")] += 1
    return len(images), shape_counts, label_counts

if not CVAT_PATH.strip():
    print("BỎ QUA: chưa chọn file CVAT. Đây là bước tùy chọn.")
else:
    image_count, shape_counts, label_counts = summarize_cvat(read_cvat_xml(CVAT_PATH))
    print("Số image trong export:", image_count)
    print("Số hình theo loại:", dict(sorted(shape_counts.items())))
    print("Số hình theo nhãn:", dict(sorted(label_counts.items())))
    print("Đây là số đếm cấu trúc, không phải kết quả QC hay điểm chất lượng.")


## 5 · Hoàn tất

Trước khi nộp, mở lại `submission/45_sampling_plan.csv`, `45_review_plan.md` và `46_gold_set_plan.md`: có đủ bốn camera × normal/hard, ngân sách 200 frame, lý do chọn, ca đề xuất cho gold set theo camera, cách phân xử/làm mới và **giới hạn của tình huống giả lập** chưa? Sau đó tiếp tục checklist và lệnh kiểm tra của repo Student. Không nộp notebook thay cho export CVAT hoặc các hiện vật trong `submission/`.
